In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, ConcatDataset
from torch_geometric.loader import DataLoader as GeometricDataLoader
from torch_geometric.data import Data
from torch_geometric.utils import dense_to_sparse, subgraph, k_hop_subgraph
import torch_geometric.nn as pyg_nn
from scipy.io import loadmat, savemat
import os
import matplotlib.pyplot as plt


In [10]:
# Load saved beta data from mat file
analysis_type = 'N170'
curr_dir = os.getcwd()
datain_dir = os.path.join(os.getcwd(), 'datain')
datain_mat = loadmat(os.path.join(datain_dir, analysis_type + '_original_betas.mat'))
binatry_matrix_mat = loadmat(os.path.join(datain_dir, 'adjacency_matrix.mat'))
datain = datain_mat['data']
binatry_matrix = binatry_matrix_mat['adjacency_matrix']

datain = np.array(datain)          # shape: [nBeta, nTime, nChan, nSubject] (adjust as needed)
binatry_matrix = np.array(binatry_matrix)

# Parse shapes
nBeta     = datain.shape[0]
nTime     = datain.shape[1]
nChan     = datain.shape[2]
nSubject  = datain.shape[3]

In [6]:
print(f"nBeta: {nBeta}, nTime: {nTime}, nChan: {nChan}, nSubject: {nSubject}")

nBeta: 26, nTime: 201, nChan: 30, nSubject: 36


In [4]:
# ===========================================
# 2) Set device & convert adjacency
# ===========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_index, _ = dense_to_sparse(torch.tensor(binatry_matrix, dtype=torch.float32))
print(device)

# ===========================================
# 3) Dataset builder
# ===========================================
def build_concat_dataset_for_beta(datain, beta_idx):
    """
    Build a ConcatDataset across all subjects for one beta index.
    `datain` is [nBeta, nTime, nChan, nSubject].
    Returns: ConcatDataset of length nSubject.
    """
    subject_datasets = []
    for s in range(nSubject):
        # Extract slice for one subject and one beta
        # shape: [nTime, nChan]
        single_beta_data = datain[beta_idx, :, :, s]

        # If each node is a channel and its feature dimension is time,
        # we might want single_beta_data to be [nChan, nTime].
        # If so, transpose here:
        single_beta_data = single_beta_data.T  # shape: [nChan, nTime]

        # For an autoencoder, features=labels
        x_tensor = torch.tensor(single_beta_data, dtype=torch.float32)
        x_tensor = x_tensor.unsqueeze(0)

        subject_datasets.append(TensorDataset(x_tensor, x_tensor))

    return ConcatDataset(subject_datasets)

cuda


In [5]:
# ===========================================
# 4) PyG Graph Dataset
# ===========================================
class EEGGraphDataset(torch.utils.data.Dataset):
    """
    Wraps a standard dataset so that each item is a PyG Data object:
      x -> node features (channels)
      edge_index -> adjacency
    """
    def __init__(self, eeg_data, edge_index):
        self.eeg_data = eeg_data
        self.edge_index = edge_index

    def __len__(self):
        return len(self.eeg_data)

    def __getitem__(self, idx):
        # item: (x, x) because it's autoencoder
        sample = self.eeg_data[idx]
        x = sample[0]  # shape: [nChan, nTime] (if you used transpose above)
        # Convert x -> float32, build PyG Data
        graph_data = Data(x=torch.tensor(x, dtype=torch.float32),
                          edge_index=self.edge_index)
        return graph_data


In [6]:
# ===========================================
# 5) Define GraphAutoencoder
# ===========================================
class GraphAutoencoder(nn.Module):
    def __init__(self, num_features, embedding_dim=64):
        """
        num_features = dimension of each node's feature vector.
                       If each channel is a node, and you pass
                       [nChan, nTime], then num_features = nTime.
        """
        super(GraphAutoencoder, self).__init__()
        # Example with two ARMAConv layers
        self.encoder_gcn1 = pyg_nn.ARMAConv(num_features, 128, 
                                            num_stacks=2, 
                                            num_layers=3,
                                            shared_weights=True)
        self.encoder_gcn2 = pyg_nn.ARMAConv(128, embedding_dim, 
                                            num_stacks=2, 
                                            num_layers=3,
                                            shared_weights=True)
        self.decoder_fc1 = nn.Linear(embedding_dim, 128)
        self.decoder_fc2 = nn.Linear(128, num_features)

    def forward(self, x, edge_index):
        x = torch.relu(self.encoder_gcn1(x, edge_index))
        latent = torch.relu(self.encoder_gcn2(x, edge_index))
        x = torch.relu(self.decoder_fc1(latent))
        reconstructed = self.decoder_fc2(x)
        return reconstructed

In [7]:
# ===========================================
# 6) Training and evaluation
# ===========================================
def train_model_all(model, train_loader, device, num_epochs):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.SmoothL1Loss()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for batch_data in train_loader:
            #print("DEBUG x.shape in train loop:", tuple(batch_data.x.shape))
            batch_data = batch_data.to(device)
            optimizer.zero_grad()

            # Forward
            reconstructed = model(batch_data.x, batch_data.edge_index)
            loss = criterion(reconstructed, batch_data.x)

            # Backprop
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)
        #print(f"Beta Model Training - Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")
    return model

def evaluation_model(model, data_sample, edge_index, device):
    """
    data_sample: shape [nChan, nTime], or [nTime, nChan],
                 whichever you used in training.
    Returns: (original_np, reconstructed_np, error_np)
    """
    model.eval()
    with torch.no_grad():
        x = torch.tensor(data_sample, dtype=torch.float32, device=device)
        eidx = edge_index.to(device)
        reconstructed = model(x, eidx)
    orig = x.cpu().numpy()
    recon = reconstructed.cpu().numpy()
    error = orig - recon
    return orig, recon, error


In [8]:
#Subject Masking functions

# Compute the reconstructed loss of other subjects
def recon_loss_others(model, dataset, edge_index, device, idx_list):
    model.eval()
    with torch.no_grad():
        edix = edge_index.to(device)
        losses = []
        for s in idx_list:
            x_s = dataset[s][0]
            x = torch.tensor(x_s, dtype=torch.float32, device=device)
            recon = model(x, edix)
            losses.append(nn.SmoothL1Loss()(recon, x).item())
    return float(np.mean(losses))

# Compute the reconstructed loss of the same subject
def recon_loss_one(model, dataset, edge_index, device, s):
    model.eval()
    with torch.no_grad():
        edix = edge_index.to(device)
        x_s = dataset[s][0]
        x = torch.tensor(x_s, dtype=torch.float32, device=device)
        recon = model(x, edix)
        loss = nn.SmoothL1Loss()(recon, x).item()
    return loss

# Build dataset excluding one subject
def build_dataset_without_subject(datain, beta_idx, s_exclude):
    subject_datasets = []
    for s in range(nSubject):
        if s == s_exclude:
            continue
        # Extract slice for one subject and one beta
        # shape: [nTime, nChan]
        single_beta_data = datain[beta_idx, :, :, s]

        # If each node is a channel and its feature dimension is time,
        # we might want single_beta_data to be [nChan, nTime].
        # If so, transpose here:
        single_beta_data = single_beta_data.T  # shape: [nChan, nTime]

        # For an autoencoder, features=labels
        x_tensor = torch.tensor(single_beta_data, dtype=torch.float32)
        x_tensor = x_tensor.unsqueeze(0)

        subject_datasets.append(TensorDataset(x_tensor, x_tensor))

    return ConcatDataset(subject_datasets)

# Retrain model without one subject
def retrain_without_one_subject(
    model_cons, detain, beta_idx, 
    edge_index, device, num_epochs, 
    batch_size, s_exclude):
    
    dataset_wo = build_dataset_without_subject(detain, beta_idx, s_exclude)
    graph_dataset_wo = EEGGraphDataset(dataset_wo, edge_index)
    train_loader_wo = GeometricDataLoader(graph_dataset_wo, batch_size=batch_size, shuffle=True)
    model_wo = model_cons().to(device)
    model_wo = train_model_all(model_wo, train_loader_wo, device, num_epochs=num_epochs)
    return model_wo

# Apply subject masking in each beta iteration
def apply_subject_masking(
    model_all, model_cons, detain, beta_idx, 
    full_dataset_i, edge_index, device, 
    num_epochs=200, batch_size=4):
    
    nSubject = len(full_dataset_i)
    base_self = np.zeros(nSubject)
    base_others = np.zeros(nSubject)
    losses_self = np.zeros(nSubject)
    losses_others = np.zeros(nSubject)
    
    for s in range(nSubject):
        print(f"Processing Subject {s+1}/{nSubject}")
        base_self[s] = recon_loss_one(model_all, full_dataset_i, edge_index, device, s)
        ind_others = [i for i in range(nSubject) if i != s]
        base_others[s] = recon_loss_others(model_all, full_dataset_i, edge_index, device, ind_others)
        
        model_wo = retrain_without_one_subject(model_cons, detain, beta_idx, edge_index, device, num_epochs, batch_size, s)
        wo_self = recon_loss_one(model_wo, full_dataset_i, edge_index, device, s)
        losses_self[s] = wo_self - base_self[s]
        
        wo_others = recon_loss_others(model_wo, full_dataset_i, edge_index, device, ind_others)
        losses_others[s] = wo_others - base_others[s]
    
    return losses_self, losses_others
    
        

In [9]:
num_features = nTime   # if each channel is a node, and feature vector = [nTime]
num_epochs   = 200     # number of training - set as needed
batch_size   = 4;      # dataset/batch to update hyperparameters

models = []
losses_self_list = []
losses_others_list = []

for iBeta in range(nBeta):
    print(f"\n=== Building dataset/model for Beta {iBeta} ===")
    # Build dataset & loader
    full_dataset_i  = build_concat_dataset_for_beta(datain, iBeta)
    graph_dataset_i = EEGGraphDataset(full_dataset_i, edge_index)
    full_loader_i   = GeometricDataLoader(graph_dataset_i, batch_size=batch_size, shuffle=True)

    # Init model & train
    model_i = GraphAutoencoder(num_features=num_features, embedding_dim=32)
    model_i = train_model_all(model_i, full_loader_i, device, num_epochs=num_epochs)
    models.append(model_i)
    
    model_cons = lambda: GraphAutoencoder(num_features=num_features, embedding_dim=32)
    
    losses_self, losses_others = apply_subject_masking(
        model_i, model_cons, datain, 
        iBeta, full_dataset_i, edge_index, 
        device, num_epochs=num_epochs, batch_size=batch_size)
    
    losses_self_list.append(losses_self)
    losses_others_list.append(losses_others)
    
all_losses_self = np.stack(losses_self_list, axis=0)
all_losses_others = np.stack(losses_others_list, axis=0)


=== Building dataset/model for Beta 0 ===


C:\Users\86136\AppData\Local\Temp\ipykernel_54844\1382389434.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  graph_data = Data(x=torch.tensor(x, dtype=torch.float32),


Processing Subject 1/36


C:\Users\86136\AppData\Local\Temp\ipykernel_54844\1187601921.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x_s, dtype=torch.float32, device=device)


Processing Subject 2/36
Processing Subject 3/36
Processing Subject 4/36
Processing Subject 5/36
Processing Subject 6/36
Processing Subject 7/36
Processing Subject 8/36
Processing Subject 9/36
Processing Subject 10/36
Processing Subject 11/36
Processing Subject 12/36
Processing Subject 13/36
Processing Subject 14/36
Processing Subject 15/36
Processing Subject 16/36
Processing Subject 17/36
Processing Subject 18/36
Processing Subject 19/36
Processing Subject 20/36
Processing Subject 21/36
Processing Subject 22/36
Processing Subject 23/36
Processing Subject 24/36
Processing Subject 25/36
Processing Subject 26/36
Processing Subject 27/36
Processing Subject 28/36
Processing Subject 29/36
Processing Subject 30/36
Processing Subject 31/36
Processing Subject 32/36
Processing Subject 33/36
Processing Subject 34/36
Processing Subject 35/36
Processing Subject 36/36

=== Building dataset/model for Beta 1 ===
Processing Subject 1/36
Processing Subject 2/36
Processing Subject 3/36
Processing Subject

In [11]:
# save importances results
save_dir = os.path.join(curr_dir, analysis_type, 'Ablation')
os.makedirs(save_dir, exist_ok=True)

save_subdir_subject = os.path.join(save_dir, 'Subject Ablation')
os.makedirs(save_subdir_subject, exist_ok=True)

imp_self = {'all_losses_self': all_losses_self}
imp_others = {'all_losses_others': all_losses_others}


savemat(os.path.join(save_subdir_subject, 'imp_self.mat'), imp_self)
savemat(os.path.join(save_subdir_subject, 'imp_others.mat'), imp_others)

In [11]:
save_dir = os.path.join(curr_dir, analysis_type, 'Ablation')
save_subdir_subject = os.path.join(save_dir, 'Subject Ablation')
print(save_subdir_subject)
imp_self_loaded = loadmat(os.path.join(save_subdir_subject, 'imp_self.mat'))
imp_others_loaded = loadmat(os.path.join(save_subdir_subject, 'imp_others.mat'))
all_losses_self = imp_self_loaded['all_losses_self']
all_losses_others = imp_others_loaded['all_losses_others']
print("shape of all_losses_self:", all_losses_self.shape)

e:\MThesis\GAEtesting\N170\Ablation\Subject Ablation
shape of all_losses_self: (5, 39)


In [13]:
# Visualize importances of subject masking
x = np.arange(1, nSubject + 1)
width = 0.35

global_max = max(np.max(all_losses_self), np.max(all_losses_others))
global_min = min(np.min(all_losses_self), np.min(all_losses_others))

for iBeta in range(nBeta):
    value_self = all_losses_self[iBeta]    # shape: [nSubject]
    value_others = all_losses_others[iBeta]

    fig, ax = plt.subplots(figsize=(10, 6))

    # Two groups of side-by-side bar plots
    bars_self = ax.bar(x - width/2, value_self, width, label='Self loss')
    bars_others = ax.bar(x + width/2, value_others, width, label='Others loss')

    # 0 reference line
    ax.axhline(0, color='black', linewidth=1)
    for xi in x:
        ax.axvline(x=xi, color='gray', linestyle='--', alpha=0.1)

    ax.set_xticks(x)
    ax.set_xticklabels(np.arange(1, nSubject + 1), fontsize=10)
    ax.tick_params(axis='y', labelsize=12)

    ax.set_xlabel('Subject Index', fontsize=16)
    ax.set_ylabel('Loss', fontsize=16)
    ax.set_ylim(global_min, global_max)
    ax.set_title(f'{analysis_type} Beta {iBeta+1} – Subject Ablation',
                 fontsize=18, pad=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    ax.legend(frameon=False, fontsize=20, loc='upper right')

    fig.tight_layout()
    fig.savefig(os.path.join(save_subdir_subject, f'{analysis_type}_Beta_{iBeta+1}_Subject_Importance.png'), dpi=300)
    #plt.show()
    plt.close(fig)
    


In [12]:
# normalize self
norm_self = np.zeros_like(all_losses_self)
for iBeta in range(nBeta):
    row = all_losses_self[iBeta]
    max_abs = np.max(np.abs(row))
    norm_self[iBeta] = row / max_abs if max_abs > 0 else row

agg_self = norm_self.mean(axis=0)

# normalize others
norm_others = np.zeros_like(all_losses_others)
for iBeta in range(nBeta):
    row = all_losses_others[iBeta]
    max_abs = np.max(np.abs(row))
    norm_others[iBeta] = row / max_abs if max_abs > 0 else row

agg_others = norm_others.mean(axis=0)

# x-axis subject indices
if str(analysis_type).lower() == 'p3':
    missing = {1, 5, 8, 37}
    x = np.array([i for i in range(1, 41) if i not in missing])
    assert len(x) == agg_self.shape[0] == agg_others.shape[0], \
        f"Length mismatch: x={len(x)}, agg_self={len(agg_self)}, agg_others={len(agg_others)}"
elif str(analysis_type).lower() == 'n170':
    missing = {4}
    x = np.array([i for i in range(1, 41) if i not in missing])
    assert len(x) == agg_self.shape[0] == agg_others.shape[0], \
        f"Length mismatch: x={len(x)}, agg_self={len(agg_self)}, agg_others={len(agg_others)}"
else:
    x = np.arange(1, nSubject + 1)
    
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars_self = ax.bar(x - width/2, agg_self, width, label='Self loss')
bars_others = ax.bar(x + width/2, agg_others, width, label='Others loss')

y_min = min(agg_self.min(), agg_others.min(), 0)
y_max = max(agg_self.max(), agg_others.max(), 0)
ax.set_ylim(y_min - 0.05*(y_max - y_min), y_max + 0.05*(y_max - y_min))

ax.axhline(0, color='black', linewidth=1)
for xi in x:
        ax.axvline(x=xi, color='gray', linestyle='--', alpha=0.1)

ax.set_xticks(x)
ax.set_xticklabels(x, fontsize=10)
ax.tick_params(axis='y', labelsize=12)

ax.set_xlabel('Subject Index', fontsize=16)
ax.set_ylabel('Aggregated loss', fontsize=16)
ax.set_title(f'{analysis_type} – Aggregated Subject Ablation',
             fontsize=18, pad=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)

ax.legend(frameon=False, fontsize=20, loc='upper right')

fig.tight_layout()
fig.savefig(
    os.path.join(
        save_subdir_subject,
        f'{analysis_type}_Aggregated_Subject_Ablation_Self_vs_Others.png'
    ),
    dpi=300
)
plt.close(fig)
